# 13 - Geometric Height as a Calibrated Low/High Score  (Track B)

Notebook 12 showed the geometric height is **accurate for tall objects**
(pole ~4% error) but **overestimates flat/low objects ~5x**. This is a
fundamental monocular ambiguity, not a fixable formula: a flat object's box
top is a *farther ground point*, and a box alone cannot separate "108 cm tall"
from "12 cm tall but extends along the ground". A ground-plane recompute is
algebraically identical to `H = Z*box_px/f` under the USS distance, so it does
not help.

But the per-class medians from nb12 show the geometric height still **orders**
objects correctly (flat 180-580 mm, tall 850-1700 mm) - it is a biased-but-
monotonic **score**. So instead of treating it as true height for the drive-
over decision, we **calibrate the Low/High threshold on labeled data** and
measure the achievable accuracy. This lets the geometric quantity act as a
class-agnostic Low/High signal - useful precisely for objects with no class.

Two results reported honestly:
1. **Metric height** (kept from nb12): accurate for tall, biased for flat;
   pole is the physics-validation figure.
2. **Calibrated Low/High score**: threshold fit on a calibration split,
   evaluated on a held-out split - balanced accuracy, ROC AUC, confusion.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vision_uss_research.alignment.height import (KNOWN_HEIGHT_MM, calibrate_low_threshold,
                                                 estimate_focal, estimate_height_mm,
                                                 height_error, roc_auc)
from vision_uss_research.csv_io import read_csv_tolerant
from vision_uss_research.labeling.boxes_io import load_boxes_tolerant
from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run
from vision_uss_research.sequences import frames_window_id

SEED = 0
CALIB_FRAC = 0.6          # sequences to FIT the threshold; rest held out
MIN_BOX_PX = 8
DISTANCE_RUN = None

# class -> low/high truth (same map as notebooks 07-09)
CLASS_BINS = {"curbstone": "low", "speedbump": "low", "woodenboard": "low",
              "pole": "high", "dummychild": "high", "bicyclestand": "high",
              "hose": "low", "step": "low", "cone": "high", "bollard": "high",
              "car": "high", "tree": "high", "bush": "high", "stonelarge": "high",
              "stonemiddle": "high", "cubestandard": "high", "standardcube": "high",
              "ubarrier": "high", "fence": "high", "curbstone_side": "low"}

dist_run = (ARTIFACTS / "alignment" / DISTANCE_RUN) if DISTANCE_RUN else resolve_run(
    ARTIFACTS / "alignment", "alignment", require_file="frame_distances.csv")
assert dist_run is not None, "run notebook 11 first"
SEQ_QUALITY = dist_run / "sequence_distance_quality.csv"

RUN_DIR = ARTIFACTS / "alignment" / new_run_id("alignment", "lowhigh_score")
RUN_DIR.mkdir(parents=True, exist_ok=True)
start_run(RUN_DIR, config={"calib_frac": CALIB_FRAC, "seed": SEED},
          inputs=[f"alignment/{dist_run.name}"], repo_root=REPO_DIR)
print(f"distance run: {dist_run.name}\noutput      : {RUN_DIR}")

## Join distance to boxes, estimate height (same as nb12)

In [ ]:
fd = pd.read_csv(dist_run / "frame_distances.csv")
fd["distance_mm"] = pd.to_numeric(fd["distance_mm"], errors="coerce")
if SEQ_QUALITY.exists():
    q = read_csv_tolerant(SEQ_QUALITY, list(pd.read_csv(SEQ_QUALITY, nrows=0).columns))
    accepted = set(q[pd.to_numeric(q["accepted"], errors="coerce") == 1]
                   ["sequence_id"].astype(str))
    fd = fd[fd["sequence_id"].astype(str).isin(accepted)]

WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))
boxes_csvs = []
for base in [ARTIFACTS / "labels" / "runs", ARTIFACTS / "labels" / "clicks"]:
    for run_dir in sorted(base.glob("*")) if base.exists() else []:
        if (run_dir / "propagated_boxes.csv").exists():
            boxes_csvs.append(run_dir / "propagated_boxes.csv")
boxes = pd.concat([load_boxes_tolerant(p) for p in boxes_csvs], ignore_index=True)
boxes = boxes[boxes["x0"].notna()].copy()
boxes["box_px_height"] = (boxes["y1"] - boxes["y0"]).abs()
boxes = (boxes.sort_values("box_px_height", ascending=False)
         .drop_duplicates(["sequence_id", "camera", "frame_idx"], keep="first"))

joined = fd.merge(boxes[["sequence_id", "camera", "frame_idx", "target_object",
                         "box_px_height"]],
                  on=["sequence_id", "camera", "frame_idx"], how="inner")
joined = joined[(joined["box_px_height"] >= MIN_BOX_PX) & joined["distance_mm"].notna()]

# focal from anchors (physics validation, kept from nb12)
anc = joined[joined["target_object"].isin(KNOWN_HEIGHT_MM)].copy()
anc["real_mm"] = anc["target_object"].map(KNOWN_HEIGHT_MM)
focal = estimate_focal(anc["box_px_height"], anc["distance_mm"], anc["real_mm"])
anc = anc.assign(pred_mm=estimate_height_mm(anc["box_px_height"], anc["distance_mm"], focal))
perr = height_error(anc[anc.target_object=="pole"]["pred_mm"],
                    anc[anc.target_object=="pole"]["real_mm"])
print(f"focal: {focal:.1f} px | pole height rel-err: "
      f"{100*perr['mean_rel_err']:.1f}% (physics check)")

joined["height_mm"] = estimate_height_mm(joined["box_px_height"],
                                         joined["distance_mm"], focal)
# one score per sequence = median height (the deployment unit)
seq = (joined.assign(cbin=joined["target_object"].str.lower().map(CLASS_BINS))
       .dropna(subset=["cbin", "height_mm"])
       .groupby(["sequence_id", "target_object", "cbin"])
       .agg(height_mm=("height_mm", "median")).reset_index())
seq["is_low"] = (seq["cbin"] == "low").astype(int)
print(f"sequences with a score + label: {len(seq)} "
      f"(low={int(seq['is_low'].sum())}, high={int((seq['is_low']==0).sum())})")

## Calibrate the Low/High threshold on a split, evaluate held-out

In [ ]:
rng = np.random.default_rng(SEED)
ids = sorted(seq["sequence_id"].astype(str).unique())
rng.shuffle(ids)
n_cal = int(CALIB_FRAC * len(ids))
cal_ids, test_ids = set(ids[:n_cal]), set(ids[n_cal:])
cal = seq[seq["sequence_id"].astype(str).isin(cal_ids)]
test = seq[seq["sequence_id"].astype(str).isin(test_ids)]

fit = calibrate_low_threshold(cal["height_mm"], cal["is_low"])
T = fit["threshold_mm"]
print(f"calibrated threshold: {T:.0f} mm  "
      f"(fit on {len(cal)} seqs, balanced acc {fit['balanced_accuracy']:.3f}, "
      f"AUC {fit['auc']:.3f})")

# evaluate on held-out
test = test.assign(geo_bin=np.where(test["height_mm"] < T, "low", "high"))
tl, th = test["is_low"] == 1, test["is_low"] == 0
recall_low = (test.loc[tl, "geo_bin"] == "low").mean()
recall_high = (test.loc[th, "geo_bin"] == "high").mean()
bacc = (recall_low + recall_high) / 2
auc = roc_auc(-test["height_mm"], test["is_low"])
print(f"\nHELD-OUT ({len(test)} sequences):")
print(f"  balanced accuracy : {bacc:.3f}")
print(f"  low recall        : {recall_low:.3f}   (drive-over objects found)")
print(f"  high recall       : {recall_high:.3f}")
print(f"  ROC AUC           : {auc:.3f}   (threshold-independent separability)")

conf = pd.crosstab(test["cbin"], test["geo_bin"],
                   rownames=["class"], colnames=["geometry"])
display(conf)

# for reference: the naive 250 mm threshold on the same held-out set
naive = np.where(test["height_mm"] < 250, "low", "high")
naive_bacc = (((naive[tl.values] == "low").mean()
               + (naive[th.values] == "high").mean()) / 2)
print(f"\n(naive 250 mm threshold balanced acc: {naive_bacc:.3f} "
      f"-> calibration lifts it to {bacc:.3f})")

In [ ]:
# score distributions with the calibrated threshold
fig, ax = plt.subplots(figsize=(9, 4))
for lab, color in [("low", "tab:green"), ("high", "tab:red")]:
    vals = seq[seq["cbin"] == lab]["height_mm"]
    ax.hist(vals, bins=40, alpha=0.5, label=f"class {lab} (n={len(vals)})", color=color)
ax.axvline(T, color="k", ls="--", lw=2, label=f"calibrated threshold {T:.0f} mm")
ax.axvline(250, color="gray", ls=":", lw=1, label="naive 250 mm")
ax.set_xlabel("geometric height score (mm)"); ax.set_ylabel("sequences")
ax.set_title("Geometric height score separates Low from High"); ax.legend()
ax.set_xlim(0, 2500)
fig.savefig(RUN_DIR / "score_distribution.jpg", dpi=110, bbox_inches="tight")
plt.show()

## Per-class behaviour under the calibrated threshold

In [ ]:
byobj = (seq.assign(geo_low=(seq["height_mm"] < T))
         .groupby(["target_object", "cbin"])
         .agg(n=("sequence_id", "nunique"),
              median_height_mm=("height_mm", "median"),
              called_low_frac=("geo_low", "mean"))
         .reset_index().sort_values("median_height_mm"))
display(byobj.round(2))

seq.assign(geo_bin=np.where(seq["height_mm"] < T, "low", "high")).to_csv(
    RUN_DIR / "sequence_lowhigh_score.csv", index=False)

summary = {
    "focal_px": round(float(focal), 1),
    "pole_height_rel_err": round(perr["mean_rel_err"], 3),
    "calibrated_threshold_mm": round(float(T), 1),
    "heldout_balanced_accuracy": round(float(bacc), 4),
    "heldout_low_recall": round(float(recall_low), 4),
    "heldout_high_recall": round(float(recall_high), 4),
    "auc": round(float(auc), 4),
    "naive_250_balanced_accuracy": round(float(naive_bacc), 4),
    "sequences": int(len(seq)),
}
finish_run(RUN_DIR, summary=summary)
print(_json.dumps(summary, indent=1))

## Reading the result & where this fits

- **AUC** is the honest, threshold-free measure of how separable Low/High are
  by geometry alone. High AUC with a much-better-than-naive balanced accuracy =
  the geometric score is a genuine class-agnostic Low/High signal.
- **Metric height stays a two-part story**: accurate for tall (pole ~4%),
  biased for flat - reported transparently. The *fundamental* monocular
  ambiguity on low objects is itself a finding that motivates the learned,
  appearance-based detector (99.8% Low/High), which does not share it.
- This geometric score is most valuable for **unknown objects** (no class):
  calibrate on knowns, apply to unknowns.

**Still on the plan: the fusion experiment** (`docs/v2_open_set_approach.md`,
Phase D) - combine three independent Low/High signals: the detector's
class-based output, the USS `ClassProbHigh`/`HeightProb`, and this geometric
score. That is the headline camera-USS-complementarity result and does not
depend on accurate metric height.